Import necessary packages

In [28]:
import networkx as nx
import json
from networkx.readwrite import json_graph
from collections import defaultdict
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt
from community import community_louvain
from collections import Counter

Create function to load data in a proper format

In [15]:
def parse_rfa(file_path):
    with open(file_path, 'rt', encoding='utf-8') as f:
        content = f.read()

    entries = content.strip().split("\n\n")
    data = []

    for entry in entries:
        record = {}
        for line in entry.split("\n"):
            if ":" in line:
                key, value = line.split(":", 1)
                record[key.strip()] = value.strip()
        data.append(record)

    return data

Create a function that builds the graph

In [16]:
def build_voter_graph(data):
    votes_by_candidate = defaultdict(list)

    for d in data:
        src = d.get("SRC", "").strip()
        tgt = d.get("TGT", "").strip()

        if not src or not tgt:
            continue

        vote = int(d["VOT"])

        # skip neutral votes
        if vote == 0:
            continue

        votes_by_candidate[tgt].append((src, vote))

    G = nx.Graph()

    for cand, voter_list in votes_by_candidate.items():
        for (u, vote_u), (v, vote_v) in combinations(voter_list, 2):

            pair_weight = vote_u * vote_v  # +1 agreement, -1 disagreement

            if G.has_edge(u, v):
                G[u][v]["weight"] += pair_weight
                G[u][v]["count"] += 1
            else:
                G.add_edge(u, v, weight=pair_weight, count=1)
        
    for u, v, data in G.edges(data=True):
        data["normalized"] = round(data["weight"] / data["count"], 5)

    return G

Build the graph and check for amount of nodes and edges

In [17]:
data = parse_rfa("wiki-RfA.txt")
G = build_voter_graph(data)

print("Voter graph:")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print(list(G.edges(data=True))[:10])

Voter graph:
Nodes: 10283
Edges: 3055279
[('Steel1943', 'Cuchullain', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'INeverCry', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'Cncmaster', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'Miniapolis', {'weight': 0, 'count': 2, 'normalized': 0.0}), ('Steel1943', 'Sven Manguard', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'Ramaksoud2000', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'RockMagnetist', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'Carrite', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'Someguy1221', {'weight': 1, 'count': 1, 'normalized': 1.0}), ('Steel1943', 'Secret', {'weight': 0, 'count': 2, 'normalized': 0.0})]


Saves graph as .json file

In [21]:
proj = json_graph.node_link_data(G)

with open("voter_graph.json", "w") as f:
    json.dump(proj, f)

Amount of edges appears to be far too big. Checking for distribution and percentiles of counts

In [20]:
# distribution of counts
counts = [d["count"] for _, _, d in G.edges(data=True)]

print("Min:", min(counts))
print("Max:", max(counts))
print("Average:", sum(counts)/len(counts))

# percentiles
counts = np.array([d["count"] for _, _, d in G.edges(data=True)])

print("Percentiles:")
for p in [50, 75, 80, 90, 95, 99]:
    print(f"{p}th percentile:", np.percentile(counts, p))

Min: 1
Max: 672
Average: 3.5340612101218905
Percentiles:
50th percentile: 1.0
75th percentile: 3.0
80th percentile: 4.0
90th percentile: 7.0
95th percentile: 12.0
99th percentile: 35.0


Appears to be a lot of noise among the edges with a small number of co-votes. Decides to create a threshold of 4 minimum co-votes to create an edges to avoid creating a graph that is too big and noisy

Creates 2 filtered graphs: G_agree for co-voters with mostly agreement, and G_disagree for co-voters with mostly disagreement

In [23]:
G_agree = nx.Graph()
G_disagree = nx.Graph()

for u, v, data in G.edges(data=True):
    c = data["count"]
    n = data["normalized"]
    w = data["weight"]

    # Keep only edges with enough evidence
    if c < 4:
        continue

    # Agreement graph
    if n > 0:
        G_agree.add_edge(
            u,
            v,
            count=c,
            weight=w,
            normalized=n,
            community_weight=n
        )

    # Disagreement graph
    elif n < 0:
        G_disagree.add_edge(
            u,
            v,
            count=c,
            weight=abs(w),
            normalized=n,
            community_weight=abs(n)
        )

print("Agree graph:")
print("Nodes:", G_agree.number_of_nodes())
print("Edges:", G_agree.number_of_edges())

print("\nDisagree graph:")
print("Nodes:", G_disagree.number_of_nodes())
print("Edges:", G_disagree.number_of_edges())

Agree graph:
Nodes: 5106
Edges: 498430

Disagree graph:
Nodes: 4317
Edges: 105995


Save agreement graph

In [26]:
agree = json_graph.node_link_data(G_agree)

with open("agree_graph.json", "w", encoding="utf-8") as f:
    json.dump(agree, f)

Save disagreement graph

In [27]:
disagree = json_graph.node_link_data(G_disagree)

with open("disagree_graph.json", "w", encoding="utf-8") as f:
    json.dump(disagree, f)

Community detection for agreement graph with a fixed random seed for reproducibility

In [34]:
partition_agree = community_louvain.best_partition(
    G_agree,
    weight="community_weight",
    random_state=42
)

Basic summary

In [35]:
num_communities = len(set(partition_agree.values()))
print("Number of communities:", num_communities)

community_sizes = Counter(partition_agree.values())
print("\nLargest communities:")
for comm, size in community_sizes.most_common(10):
    print(f"Community {comm}: {size} nodes")

Number of communities: 6

Largest communities:
Community 5: 1694 nodes
Community 2: 1318 nodes
Community 3: 1085 nodes
Community 0: 991 nodes
Community 4: 14 nodes
Community 1: 4 nodes


Attach community labels to the graph

In [36]:
nx.set_node_attributes(G_agree, partition_agree, "community")

Save graph with communites

In [38]:
com = json_graph.node_link_data(G_agree)

with open("agree_graph_communities.json", "w", encoding="utf-8") as f:
    json.dump(com, f)